# Librerias


In [48]:
!pip install yfinance --quiet
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path

In [81]:
!pip install plotly==5.24.1 --quiet

# Drive


In [49]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ruta_base = Path("/content/drive/MyDrive/Colab Notebooks/data/raw")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Extract: precios Chicago (yfinance)

In [50]:
tickers = {"soja": "ZS=F", "maiz": "ZC=F", "trigo": "ZW=F"}
precios = {}
for cultivo, ticker in tickers.items():
    df = yf.download(ticker, start="2015-01-01", end="2024-12-31")
    precios[cultivo] = df
    print(f"{cultivo}: {df.shape[0]} filas descargadas")

/tmp/ipykernel_1408/1185501102.py:4: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_1408/1185501102.py:4: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_1408/1185501102.py:4: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

soja: 2513 filas descargadas
maiz: 2511 filas descargadas
trigo: 2513 filas descargadas


# Extract: producción MAGyP (series nacionales)

In [51]:
ruta_base = Path("/content/drive/MyDrive/PROYECTO UNO")

produccion = {}
archivos_nacionales = {
    "trigo": ruta_base / "trigo-serie-1923-2025-anual.csv",
    "maiz": ruta_base / "maiz-serie-1923-2024-anual.csv",
    "soja": ruta_base / "soja-serie-1941-2024-anual.csv",
}

for cultivo, ruta in archivos_nacionales.items():
    if not ruta.exists():
        raise FileNotFoundError(f"No encontré {ruta.name}")
    produccion[cultivo] = pd.read_csv(ruta)
    print(f"\n{cultivo.upper()}: {produccion[cultivo].shape}")
    print(produccion[cultivo].columns.tolist())
    print(produccion[cultivo].head(3))


TRIGO: (103, 5)
['indice_tiempo', 'superficie_sembrada_trigo_ha', 'superficie_cosechada_trigo_ha', 'produccion_trigo_t', 'rendimiento_trigo_kgxha']
   indice_tiempo  superficie_sembrada_trigo_ha  superficie_cosechada_trigo_ha  \
0           1923                       6833343                        6778430   
1           1924                       7200500                        6465440   
2           1925                       6577970                        5928650   

   produccion_trigo_t  rendimiento_trigo_kgxha  
0             6635565                   978.63  
1             5201979                   805.14  
2             4093998                   691.22  

MAIZ: (102, 5)
['indice_tiempo', 'superficie_sembrada_maiz_ha', 'superficie_cosechada_maiz_ha', 'produccion_maiz_t', 'rendimiento_maiz_kgxha']
   indice_tiempo  superficie_sembrada_maiz_ha  superficie_cosechada_maiz_ha  \
0           1923                      3435430                           NaN   
1           1924            

# EDA

## ¿Qué rango temporal cubre cada fuente, hay nulos?

In [52]:
print("=== PRECIOS (yfinance) ===")
for cultivo, df in precios.items():
    print(f"\n{cultivo.upper()}")
    print(f"Rango de fechas: {df.index.min().date()} a {df.index.max().date()}")
    print(f"Filas totales: {len(df)}")
    print(f"Nulos por columna:\n{df.isna().sum()}")

print("\n\n=== PRODUCCIÓN (MAGyP) ===")
for cultivo, df in produccion.items():
    print(f"\n{cultivo.upper()}")
    print(f"Rango de años: {df['indice_tiempo'].min()} a {df['indice_tiempo'].max()}")
    print(f"Filas totales: {len(df)}")
    print(f"Nulos por columna:\n{df.isna().sum()}")

=== PRECIOS (yfinance) ===

SOJA
Rango de fechas: 2015-01-02 a 2024-12-30
Filas totales: 2513
Nulos por columna:
Price   Ticker
Close   ZS=F      0
High    ZS=F      0
Low     ZS=F      0
Open    ZS=F      0
Volume  ZS=F      0
dtype: int64

MAIZ
Rango de fechas: 2015-01-02 a 2024-12-30
Filas totales: 2511
Nulos por columna:
Price   Ticker
Close   ZC=F      0
High    ZC=F      0
Low     ZC=F      0
Open    ZC=F      0
Volume  ZC=F      0
dtype: int64

TRIGO
Rango de fechas: 2015-01-02 a 2024-12-30
Filas totales: 2513
Nulos por columna:
Price   Ticker
Close   ZW=F      0
High    ZW=F      0
Low     ZW=F      0
Open    ZW=F      0
Volume  ZW=F      0
dtype: int64


=== PRODUCCIÓN (MAGyP) ===

TRIGO
Rango de años: 1923 a 2025
Filas totales: 103
Nulos por columna:
indice_tiempo                    0
superficie_sembrada_trigo_ha     0
superficie_cosechada_trigo_ha    0
produccion_trigo_t               0
rendimiento_trigo_kgxha          0
dtype: int64

MAIZ
Rango de años: 1923 a 2024
Filas to

## ¿Cómo se ve la calidad SOLO en el rango que nos importa (2015-2024)?

In [53]:
print("=== PRODUCCIÓN 2015-2024 — nulos y estadísticas ===")
for cultivo, df in produccion.items():
    recorte = df[(df["indice_tiempo"] >= 2015) & (df["indice_tiempo"] <= 2024)]
    print(f"\n{cultivo.upper()} — {len(recorte)} años disponibles en el rango")
    print(f"Nulos: {recorte.isna().sum().sum()}")
    print(recorte.describe())

=== PRODUCCIÓN 2015-2024 — nulos y estadísticas ===

TRIGO — 10 años disponibles en el rango
Nulos: 0
       indice_tiempo  superficie_sembrada_trigo_ha  \
count       10.00000                  1.000000e+01   
mean      2019.50000                  6.176683e+06   
std          3.02765                  7.340584e+05   
min       2015.00000                  4.381128e+06   
25%       2017.25000                  5.919411e+06   
50%       2019.50000                  6.325582e+06   
75%       2021.75000                  6.684966e+06   
max       2024.00000                  6.951171e+06   

       superficie_cosechada_trigo_ha  produccion_trigo_t  \
count                   1.000000e+01        1.000000e+01   
mean                    5.847041e+06        1.741786e+07   
std                     7.975719e+05        3.314501e+06   
min                     3.953102e+06        1.131495e+07   
25%                     5.568627e+06        1.630093e+07   
50%                     5.936563e+06        1.84526

## ¿Hay duplicados o precios en cero/negativos (imposibles)?

In [14]:
print("=== PRECIOS — valores imposibles ===")
for cultivo, df in precios.items():
    duplicados = df.index.duplicated().sum()
    precios_invalidos = (df["Close"] <= 0).sum()
    print(f"{cultivo}: {duplicados} fechas duplicadas, {precios_invalidos} precios <= 0")

=== PRECIOS — valores imposibles ===
soja: 0 fechas duplicadas, Ticker
ZS=F    0
dtype: int64 precios <= 0
maiz: 0 fechas duplicadas, Ticker
ZC=F    0
dtype: int64 precios <= 0
trigo: 0 fechas duplicadas, Ticker
ZW=F    0
dtype: int64 precios <= 0


# TRANSFORM

## Aplanar el MultiIndex de cada DataFrame de precios

In [15]:
for cultivo, df in precios.items():
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    print(f"{cultivo}: columnas ahora = {df.columns.tolist()}")

soja: columnas ahora = ['Close', 'High', 'Low', 'Open', 'Volume']
maiz: columnas ahora = ['Close', 'High', 'Low', 'Open', 'Volume']
trigo: columnas ahora = ['Close', 'High', 'Low', 'Open', 'Volume']


## Quedarnos con Close, renombrar, y apilar los 3 cultivos en una sola tabla

In [54]:
precios_diarios = {}

for cultivo, df in precios.items():
    df_limpio = df[["Close"]].copy()
    df_limpio.columns = ["precio_cierre"]
    df_limpio["cultivo"] = cultivo
    df_limpio.index.name = "fecha"
    precios_diarios[cultivo] = df_limpio

precios_diarios_todos = pd.concat(precios_diarios.values())
precios_diarios_todos = precios_diarios_todos.reset_index()

print(precios_diarios_todos.shape)
print(precios_diarios_todos["cultivo"].value_counts())
precios_diarios_todos.head()

(7537, 3)
cultivo
soja     2513
trigo    2513
maiz     2511
Name: count, dtype: int64


,fecha,precio_cierre,cultivo
0,2015-01-02,1002.50,soja
1,2015-01-05,1039.75,soja
2,2015-01-06,1051.00,soja
3,2015-01-07,1052.75,soja
4,2015-01-08,1045.00,soja


## Unificar nombres de columna de los 3 cultivos (MAGyP)

In [17]:
def normalizar_produccion(df, cultivo):
    df_renombrado = df.rename(columns={
        "indice_tiempo": "anio",
        f"superficie_sembrada_{cultivo}_ha": "superficie_sembrada_ha",
        f"superficie_cosechada_{cultivo}_ha": "superficie_cosechada_ha",
        f"produccion_{cultivo}_t": "produccion_t",
        f"rendimiento_{cultivo}_kgxha": "rendimiento_kgxha",
    })
    df_renombrado["cultivo"] = cultivo
    return df_renombrado

produccion_normalizada = pd.concat([
    normalizar_produccion(produccion["trigo"], "trigo"),
    normalizar_produccion(produccion["maiz"], "maiz"),
    normalizar_produccion(produccion["soja"], "soja"),
], ignore_index=True)

print(produccion_normalizada.shape)
print(produccion_normalizada["cultivo"].value_counts())
produccion_normalizada.head()

(289, 6)
cultivo
trigo    103
maiz     102
soja      84
Name: count, dtype: int64


,anio,superficie_sembrada_ha,superficie_cosechada_ha,produccion_t,rendimiento_kgxha,cultivo
0,1923,6833343,6778430.0,6635565,978.63,trigo
1,1924,7200500,6465440.0,5201979,805.14,trigo
2,1925,6577970,5928650.0,4093998,691.22,trigo
3,1926,7800000,7669751.0,6261624,824.78,trigo
4,1927,8372990,8172990.0,7682990,1002.36,trigo


## Filtrar al rango comparable con yfinance (2015-2024)

In [55]:
produccion_2015_2024 = produccion_normalizada[
    (produccion_normalizada["anio"] >= 2015) &
    (produccion_normalizada["anio"] <= 2024)
].reset_index(drop=True)

print(produccion_2015_2024.shape)
print(produccion_2015_2024["cultivo"].value_counts())
produccion_2015_2024

(30, 6)
cultivo
trigo    10
maiz     10
soja     10
Name: count, dtype: int64


,anio,superficie_sembrada_ha,superficie_cosechada_ha,produccion_t,rendimiento_kgxha,cultivo
0,2015,4381128,3953102.0,11314952,2765.98,trigo
1,2016,6364015,5566385.0,18395106,3047.65,trigo
2,2017,5927610,5822173.0,18518045,2978.43,trigo
3,2018,6287149,6050953.0,19459727,2991.11,trigo
4,2019,6951171,6729838.0,19776942,2736.07,trigo
5,2020,6729898,6394102.0,17644277,2522.01,trigo
6,2021,6751729,6548477.0,22150287,2895.02,trigo
7,2022,5907287,5488728.0,12555860,1954.72,trigo
8,2023,5916678,5575352.0,15853154,2584.31,trigo
9,2024,6550169,6341303.0,18510280,2596.39,trigo


##  Guardar las bases limpias en data/processed

In [56]:
ruta_processed = Path("/content/drive/MyDrive/PROYECTO UNO/processed")
ruta_processed.mkdir(exist_ok=True)

precios_diarios_todos.to_csv(ruta_processed / "precios_diarios_2015_2024.csv", index=False)
produccion_2015_2024.to_csv(ruta_processed / "produccion_2015_2024.csv", index=False)

print("Guardado en:", ruta_processed)

Guardado en: /content/drive/MyDrive/PROYECTO UNO/processed


# Análisis (PREGUNTA N°1)


## Estacionalidad. ¿En qué meses históricamente sube o baja el precio?
 Para responder esto con rigor (no "a ojo"), la métrica correcta no es el precio en sí, es la variación porcentual mes a mes (el retorno mensual). ¿Por qué no el precio directo? Porque el precio de la soja en 2015 (~$1000) y en 2024 pueden estar en niveles totalmente distintos por inflación, oferta/demanda global, etc. Si promediamos el precio de enero de todos los años, se mezclan niveles distintos y el promedio no dice nada real sobre "estacionalidad". En cambio, si calculamos cuánto varió el precio dentro de cada mes (en %), esa variación sí es comparable entre 2015 y 2024, porque es relativa, no absoluta. La lógica en pasos:

- De cada fecha, extraer el mes y el año.
- Para cada combinación cultivo-año-mes, calcular el precio de cierre del último día del mes.
- Calcular la variación % respecto al mes anterior.
- Promediar esa variación % por mes (enero, febrero...) a través de todos los años → eso es la estacionalidad


### Extraer año y mes de cada fecha

In [57]:
precios_diarios_todos["anio"] = precios_diarios_todos["fecha"].dt.year
precios_diarios_todos["mes"] = precios_diarios_todos["fecha"].dt.month

precios_diarios_todos.head()

,fecha,precio_cierre,cultivo,anio,mes
0,2015-01-02,1002.50,soja,2015,1
1,2015-01-05,1039.75,soja,2015,1
2,2015-01-06,1051.00,soja,2015,1
3,2015-01-07,1052.75,soja,2015,1
4,2015-01-08,1045.00,soja,2015,1


### Quedarnos con el precio de cierre de cada mes (el último día hábil)

In [58]:
cierre_mensual = (
    precios_diarios_todos
    .sort_values("fecha")
    .groupby(["cultivo", "anio", "mes"], as_index=False)
    .last()
)

print(cierre_mensual.shape)
cierre_mensual.head()

(360, 5)


,cultivo,anio,mes,fecha,precio_cierre
0,maiz,2015,1,2015-01-30,370.00
1,maiz,2015,2,2015-02-27,384.50
2,maiz,2015,3,2015-03-31,376.25
3,maiz,2015,4,2015-04-30,362.50
4,maiz,2015,5,2015-05-29,351.50


### Calcular la variación % mes a mes, por cultivo

In [59]:
cierre_mensual = cierre_mensual.sort_values(["cultivo", "anio", "mes"])
cierre_mensual["variacion_pct"] = (
    cierre_mensual
    .groupby("cultivo")["precio_cierre"]
    .pct_change() * 100
)

cierre_mensual.head(15)

,cultivo,anio,mes,fecha,precio_cierre,variacion_pct
0,maiz,2015,1,2015-01-30,370.00,NaN
1,maiz,2015,2,2015-02-27,384.50,3.918919
2,maiz,2015,3,2015-03-31,376.25,-2.145644
3,maiz,2015,4,2015-04-30,362.50,-3.654485
4,maiz,2015,5,2015-05-29,351.50,-3.034483
5,maiz,2015,6,2015-06-30,414.00,17.780939
6,maiz,2015,7,2015-07-31,371.00,-10.386473
7,maiz,2015,8,2015-08-31,363.75,-1.954178
8,maiz,2015,9,2015-09-30,387.75,6.597938
9,maiz,2015,10,2015-10-30,382.25,-1.418440


### Promediar la variación % por mes, a través de todos los años


In [60]:
estacionalidad = (
    cierre_mensual
    .dropna(subset=["variacion_pct"])
    .groupby(["cultivo", "mes"], as_index=False)["variacion_pct"]
    .mean()
)

nombres_mes = {1:"Ene",2:"Feb",3:"Mar",4:"Abr",5:"May",6:"Jun",
               7:"Jul",8:"Ago",9:"Sep",10:"Oct",11:"Nov",12:"Dic"}
estacionalidad["mes_nombre"] = estacionalidad["mes"].map(nombres_mes)

print(estacionalidad.shape)
estacionalidad.sort_values(["cultivo","mes"])

(36, 4)


,cultivo,mes,variacion_pct,mes_nombre
0,maiz,1,2.391574,Ene
1,maiz,2,-0.489958,Feb
2,maiz,3,1.176098,Mar
3,maiz,4,3.353797,Abr
4,maiz,5,0.619490,May
5,maiz,6,-1.205157,Jun
6,maiz,7,-7.613143,Jul
7,maiz,8,-2.830513,Ago
8,maiz,9,5.757106,Sep
9,maiz,10,1.393662,Oct


Patrón que aparece en los tres cultivos a la vez
Julio-Agosto: caída en los tres

Maíz: Jul -7.6%, Ago -2.8%
Soja: Jul -3.3%, Ago -5.1%
Trigo: Jul -2.2%, Ago -5.2%

Septiembre: rebote fuerte en los tres

Maíz: +5.8%
Trigo: +6.2%
Soja: +0.6% (más suave, pero sigue siendo positivo)

Hallazgo: Julio-agosto es plena época de crecimiento y definición de rendimiento en el hemisferio norte (EE.UU. es el jugador que mueve el precio de referencia en Chicago) es cuando el mercado tiene más certeza sobre "cuánto va a rendir la cosecha", y si el clima acompaña, la expectativa de oferta abundante presiona el precio a la baja. Septiembre es el arranque de la cosecha real de maíz y soja en EE.UU. ahí suele haber un reacomodo del mercado una vez que la incertidumbre climática ya pasó.


### Heatmap de estacionalidad (variación % promedio por mes y cultivo)

In [24]:
import plotly.express as px

orden_meses = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]

tabla_heatmap = estacionalidad.pivot(
    index="cultivo", columns="mes_nombre", values="variacion_pct"
)[orden_meses]

fig = px.imshow(
    tabla_heatmap,
    text_auto=".1f",
    color_continuous_scale="RdYlGn",   # rojo = caída, verde = suba
    color_continuous_midpoint=0,        # el 0% es el punto neutro del color
    aspect="auto",
    labels=dict(color="Variación % promedio"),
    title="Estacionalidad de precios — Chicago 2015-2024"
)
fig.update_layout(
    xaxis_title="Mes",
    yaxis_title="Cultivo",
    height=350
)
fig.show()

## Conclusión ¿En qué meses históricamente sube o baja el precio?

Entre 2015 y 2024, los tres cultivos muestran una caída sistemática en julio-agosto (hasta -7.6% en maíz) y un rebote en septiembre (hasta +6.2% en trigo), coincidiendo con la etapa de definición de rendimiento y el inicio de cosecha en el hemisferio norte, que es donde se forma el precio de referencia de Chicago. Para un productor con capacidad de almacenamiento, este patrón sugiere que sostener la venta más allá de agosto captura, en promedio, mejor precio aunque 10 años de historia no garantizan que el patrón se repita, y la decisión sigue dependiendo de la necesidad de liquidez de cada productor.


# Análisis (PREGUNTA N°2)

## Correlación entre cultivos, si sube la soja, ¿el maíz y el trigo suben también, o se mueven independientes?

Esto importa mucho para una empresa ya que si tenemos exposición a los tres cultivos (como productor, como acopiador, o como área de riesgo), y los tres se mueven siempre juntos, El "portafolio" en realidad tiene un solo riesgo grande, no tres riesgos chicos no se estan diversificado aunque parezca que sí. Si en cambio se mueven independientes, un mal año en soja se puede compensar con un buen año en maíz. Entonces nos ayudamos con el coeficiente de correlación de Pearson, que da un número entre -1 y +1, Por definicion

Cerca de +1: cuando uno sube, el otro sube casi siempre igual (muy correlacionados)
Cerca de 0: no hay relación lineal entre ambos
Cerca de -1: cuando uno sube, el otro baja (se mueven en contra)


### Retornos diarios por cultivo

In [62]:
precios_diarios_todos = precios_diarios_todos.sort_values(["cultivo", "fecha"])
precios_diarios_todos["retorno_diario"] = (
    precios_diarios_todos
    .groupby("cultivo")["precio_cierre"]
    .pct_change() * 100
)

precios_diarios_todos.head()

,fecha,precio_cierre,cultivo,anio,mes,retorno_diario
2513,2015-01-02,395.75,maiz,2015,1,NaN
2514,2015-01-05,406.00,maiz,2015,1,2.590019
2515,2015-01-06,405.00,maiz,2015,1,-0.246305
2516,2015-01-07,396.25,maiz,2015,1,-2.160494
2517,2015-01-08,394.25,maiz,2015,1,-0.504732


### Pasar de formato largo a ancho, una columna por cultivo

In [63]:
retornos_ancho = precios_diarios_todos.pivot(
    index="fecha", columns="cultivo", values="retorno_diario"
)

print(retornos_ancho.shape)
retornos_ancho.head()

(2513, 3)


cultivo,maiz,soja,trigo
fecha,,,
2015-01-02,NaN,NaN,NaN
2015-01-05,2.590019,3.715711,1.333333
2015-01-06,-0.246305,1.081991,0.466893
2015-01-07,-2.160494,0.166508,-2.070131
2015-01-08,-0.504732,-0.736167,-2.157032


### Matriz de correlación de Pearson

In [64]:
matriz_correlacion = retornos_ancho.corr()
matriz_correlacion

cultivo,maiz,soja,trigo
cultivo,,,
maiz,1.000000,0.542417,0.535489
soja,0.542417,1.000000,0.340850
trigo,0.535489,0.340850,1.000000


### Heatmap de correlación

In [65]:
fig2 = px.imshow(
    matriz_correlacion,
    text_auto=".2f",
    color_continuous_scale="RdYlGn",
    zmin=-1, zmax=1,
    color_continuous_midpoint=0,
    aspect="auto",
    labels=dict(color="Correlación"),
    title="Correlación de retornos diarios — Soja, Maíz, Trigo (2015-2024)"
)
fig2.update_layout(height=400)
fig2.show()

## Conclusión, si sube la soja, ¿el maíz y el trigo suben también, o se mueven independientes?

Maíz-Soja: 0.54 — correlación moderada-alta
Maíz-Trigo: 0.54 — correlación moderada-alta
Soja-Trigo: 0.34 — correlación moderada, la más baja de las tres

Ninguno de los tres pares está cerca de 1.0 (movimiento casi idéntico) ni cerca de 0 (movimiento totalmente independiente) todos están en zona moderada, lo cual ya es un hallazgo en sí mismo no son el mismo riesgo repetido tres veces, pero tampoco son riesgos totalmente separados.

Lo más interesante es la diferencia entre pares maíz se correlaciona parecido tanto con soja como con trigo (0.54 y 0.54), pero soja y trigo entre sí están más despegados (0.34). Una explicacion de mercado; el maíz y la soja compiten por la misma tierra y la misma época de siembra en Argentina (misma campaña gruesa), así que factores climáticos y de costo de siembra los afectan de forma parecida. Trigo, en cambio, es un cultivo de invierno (campaña fina, se siembra y cosecha en otro momento del año) comparte menos factores estacionales directos con soja, por eso la correlación más baja.
Que hallazgos se encuentran en estos datos. Los tres cultivos muestran correlación moderada (0.34 a 0.54), sin par extremo.
Por qué maíz y soja comparten campaña y factores de oferta (0.54); trigo, al ser cultivo de invierno, se despega más de soja (0.34). Esto implica que, los tres cultivos ofrece cierta diversificación real no es un riesgo triplicado, pero tampoco hay que sobreestimar la protección, porque 0.54 sigue siendo una correlación moderada, no despreciable. Salvo que la correlación de Pearson mide relación lineal únicamente si existiera una relación más compleja (no lineal) entre los precios, este coeficiente no la captaría. Además, correlación no implica causalidad: que se muevan parecido no significa que uno cause el movimiento del otro, ambos pueden estar reaccionando al mismo factor externo (clima, dólar, demanda china, etc.).


# Análisis (PREGUNTA N°3)

## ¿La producción argentina explica algo del precio internacional, o el precio se define en otro lado sin importar cuánto produzcamos?

Mi pregunta antes de, "si hay escasez, sube el precio" es lógica de oferta y demanda básica, y es válida... pero solo si estamos hablando del mercado que realmente mueve el precio. Ahí está el punto fino de esta pregunta. El dato clave que tenemos que tener en la cabeza antes de mirar el número, Argentina es un jugador grande en soja (2do-3er exportador mundial) pero el precio de referencia se forma en Chicago, donde pesan mucho más EE.UU. (el productor más grande del mundo) y la demanda de China. Una mala o buena cosecha argentina puede mover el precio local (lo que se paga en el puerto de Rosario), pero es mucho más discutible que mueva el precio de Chicago, ese es el mercado internacional que estamos usando (ZS=F). Qué vamos a calcular, la correlación entre la producción anual argentina (o el rendimiento, que es más sensible al clima que la superficie) y el precio promedio anual de Chicago, para cada cultivo, con los 10 años que tenemos (2015-2024).
La salvedad que hay que decir ANTES de ver el resultado, no después con solo 10 años de datos, cualquier correlación que salga (alta o baja) es estadísticamente débil, 10 puntos es una muestra chica para afirmar algo con seguridad. Esto no invalida el ejercicio, pero es la primera vez en el proyecto donde el tamaño de muestra es una limitación real. Es distinto a la Pregunta 2, donde teníamos miles de días de dato diario acá tenemos 10 puntos, uno por año, porque la producción se mide una vez por campaña, no todos los días.

Qué resultado espero, y por qué importa que lo diga ANTES de calcularlo, mi hipótesis, basada en cómo funciona realmente el mercado de granos, es que la correlación va a salir baja o incluso negativa no porque el modelo esté mal, sino porque el precio de Chicago no depende principalmente de la producción argentina. Si sale así, ESE es el hallazgo real de esta pregunta, confirmar con datos que el productor argentino es "tomador de precio" (price taker), no formador. Es una conclusión más valiosa que si hubiera salido una correlación alta, porque le desarma una creencia común a alguien con datos reales.

### Precio promedio anual por cultivo

In [66]:
precio_anual = (
    precios_diarios_todos
    .groupby(["cultivo", "anio"], as_index=False)["precio_cierre"]
    .mean()
    .rename(columns={"precio_cierre": "precio_promedio_anual"})
)

print(precio_anual.shape)
precio_anual.head()

(30, 3)


,cultivo,anio,precio_promedio_anual
0,maiz,2015,376.672619
1,maiz,2016,358.454000
2,maiz,2017,359.224104
3,maiz,2018,368.066265
4,maiz,2019,383.387897


### Unir producción (MAGyP) con precio anual (yfinance)

In [31]:
produccion_precio = produccion_2015_2024.merge(
    precio_anual,
    on=["cultivo", "anio"],
    how="inner"
)

print(produccion_precio.shape)
produccion_precio[["cultivo", "anio", "produccion_t", "rendimiento_kgxha", "precio_promedio_anual"]]

(30, 7)


,cultivo,anio,produccion_t,rendimiento_kgxha,precio_promedio_anual
0,trigo,2015,11314952,2765.98,507.378968
1,trigo,2016,18395106,3047.65,436.208000
2,trigo,2017,18518045,2978.43,435.760956
3,trigo,2018,19459727,2991.11,495.534861
4,trigo,2019,19776942,2736.07,493.896825
5,trigo,2020,17644277,2522.01,549.526680
6,trigo,2021,22150287,2895.02,702.336310
7,trigo,2022,12555860,1954.72,901.670319
8,trigo,2023,15853154,2584.31,643.963000
9,trigo,2024,18510280,2596.39,572.502988


Lo que ya salta a simple vista, antes de seguir con los calculos justo en la fila de 2022 en los tres cultivos, es el año de la sequía "triple Niña" que se ijdentifico antes;

- Trigo: rendimiento más bajo del período (1954.72) → precio 901.67 (el segundo más alto de su serie)
- Maíz: rendimiento más bajo del período (4677.26) → precio 693.88 (uno de los más altos)
- Soja: rendimiento más bajo del período (1698.17) → precio 1550.66 (el más alto de toda la serie)

A simple vista parece que confirma la intuición, menos producción, más precio. Tambien historicamente se debe resaltar que en 2022 fue también el año en que empezó la guerra Rusia-Ucrania (febrero 2022), que disparó los precios de granos a nivel mundial por motivos que no tienen nada que ver con la cosecha argentina. Entonces tenemos dos causas posibles ocurriendo al mismo tiempo, y con una correlación simple no se va a poder separar cuál pesó más.

### Correlación producción/rendimiento vs precio, por cultivo

In [32]:
resultados_correlacion = []

for cultivo in produccion_precio["cultivo"].unique():
    datos_cultivo = produccion_precio[produccion_precio["cultivo"] == cultivo]
    corr_produccion = datos_cultivo["produccion_t"].corr(datos_cultivo["precio_promedio_anual"])
    corr_rendimiento = datos_cultivo["rendimiento_kgxha"].corr(datos_cultivo["precio_promedio_anual"])
    resultados_correlacion.append({
        "cultivo": cultivo,
        "corr_produccion_precio": corr_produccion,
        "corr_rendimiento_precio": corr_rendimiento
    })

resultados_correlacion = pd.DataFrame(resultados_correlacion)
resultados_correlacion

,cultivo,corr_produccion_precio,corr_rendimiento_precio
0,trigo,-0.311864,-0.815873
1,maiz,-0.080694,-0.829992
2,soja,-0.637473,-0.653829


## Conclusión, ¿la producción argentina explica algo del precio internacional, o el precio se define en otro lado sin importar cuánto produzcamos?

A primera vista, "-0.65 a -0.83 de correlación" parece decir "sí, la producción argentina mueve el precio internacional" lo contrario de lo que planteamos como hipótesis. Pero el rendimiento (kg por hectárea) no es un número exclusivo de Argentina es un indicador de cómo estuvo el clima esa campaña en la región, y muchas veces las mismas condiciones climáticas globales (sequías, La Niña, El Niño) afectan a Argentina, a Brasil y a EE.UU. al mismo tiempo o en fases relacionadas. Entonces la correlación fuerte con el rendimiento puede estar reflejando "el clima global golpeó a todos los productores importantes a la vez, y por eso subió Chicago" no necesariamente "Argentina por sí sola mueve el precio mundial".

La correlación no distingue si Argentina causa el precio, o si Argentina y el precio reaccionan al mismo factor externo (el clima regional/global). Con los datos que tenemos, no podemos separar esas dos explicaciones, necesitaríamos datos de producción de EE.UU. y Brasil también para aislar el efecto puramente argentino. Eso es una limitación real del alcance de este proyecto.

Entonces: Como hallazgo tenemos que, el rendimiento argentino correlaciona fuerte y negativamente con el precio de Chicago (-0.65 a -0.83); la producción total, más débil (-0.08 a -0.64), siendo maíz el caso donde casi no hay relación con el volumen total producido.
Por qué el rendimiento es sensible al clima, y probablemente esté capturando un factor climático regional/global compartido con otros países productores, no un efecto exclusivo de Argentina sobre el mercado mundial.

Esto implica que, Argentina sigue siendo mayormente tomador de precio no hay evidencia de que el volumen que decide sembrar/cosechar el país, por sí solo, sea lo que mueve Chicago; lo que sí parece influir es el mismo shock climático que afecta a la región agrícola global.

Con 10 años de datos y sin series de producción de EE.UU./Brasil para comparar, no se puede separar "Argentina causa el precio" de "Argentina y el precio reaccionan al mismo clima global" el hallazgo es una asociación fuerte, no una relación causal comprobada. El caso 2022 (sequía + guerra Rusia-Ucrania simultáneas) es un buen ejemplo de por qué hay que ser cautos acá.

# Análisis (PREGUNTA N°4)

## ¿Qué tan riesgoso fue cada cultivo, año a año, y qué campañas fueron las más peligrosas para el precio?

Aca necesito calcular la volatilidad, que es nada mas y nada menos que la desviación estándar de los retornos diarios mide cuánto se mueve el precio de un día a otro, en promedio, sin importar si sube o baja. Un cultivo con alta volatilidad puede tener el mismo precio promedio que uno tranquilo, pero con saltos mucho más bruscos día a día. Para un productor o un área de riesgo, esto importa porque la volatilidad es lo que hace que valga la pena o no pagar por cobertura, si el precio casi no se mueve, cubrirse con futuros es un gasto innecesario; si se mueve mucho, la cobertura protege de verdad.

Me planteo una pregunta a mi mismo, ¿Por qué la calculo por año y no un solo número para todo el período? un promedio de volatilidad 2015-2024 me taparia que 2022 (la sequía + guerra) fue mucho más riesgoso que, digamos, 2019. Necesito ver la volatilidad año a año para identificar qué campañas fueron las más turbulentas y creo que eso es lo que realmente le sirve a alguien que está por decidir si cubrirse o no en la campaña que viene.

Detalle técnico que vale la pena saber, la volatilidad diaria sola es un número chico y poco intuitivo (algo como "1.5%"). En finanzas es estándar anualizarla, multiplicándola por la raíz cuadrada de 252 (la cantidad aproximada de días hábiles de trading en un año). Esto convierte "cuánto se mueve por día" en "cuánto se movería a lo largo de un año si ese ritmo se mantuviera" es la forma en que cualquier persona de mercados de capitales va a esperar ver ese número.

### Volatilidad anualizada por cultivo y año

In [67]:
volatilidad_anual = (
    precios_diarios_todos
    .dropna(subset=["retorno_diario"])
    .groupby(["cultivo", "anio"])["retorno_diario"]
    .std()
    .reset_index()
    .rename(columns={"retorno_diario": "volatilidad_diaria"})
)

volatilidad_anual["volatilidad_anualizada"] = (
    volatilidad_anual["volatilidad_diaria"] * np.sqrt(252)
)

print(volatilidad_anual.shape)
volatilidad_anual.sort_values(["cultivo", "anio"])

(30, 4)


,cultivo,anio,volatilidad_diaria,volatilidad_anualizada
0,maiz,2015,1.497524,23.772463
1,maiz,2016,1.539231,24.434539
2,maiz,2017,1.224143,19.432666
3,maiz,2018,1.179467,18.723461
4,maiz,2019,1.555813,24.697770
5,maiz,2020,1.355691,21.520924
6,maiz,2021,2.264157,35.942385
7,maiz,2022,1.921559,30.503807
8,maiz,2023,2.008719,31.887419
9,maiz,2024,1.278708,20.298868


In [70]:
fig3 = px.line(
    volatilidad_anual,
    x="anio", y="volatilidad_anualizada", color="cultivo",
    markers=True,
    title="Volatilidad anualizada por cultivo (2015-2024)",
    labels={"volatilidad_anualizada": "Volatilidad anualizada (%)", "anio": "Año"}
)
fig3.update_layout(height=450)
fig3.show()

## Conclusión ¿qué tan riesgoso fue cada cultivo, año a año?

- Trigo: 2022 es, por lejos, el año más volátil de toda la serie 52.36%, muy por encima del segundo lugar (2023, 36.15%). Un salto brutal.
- Soja: 2022 es el máximo de la serie — 29.05%, aunque no tan despegado del resto (2021 estaba en 25.05%).
- Maíz: acá la sorpresa el máximo real es 2021 (35.94%), no 2022 (que queda tercero, 30.50%, detrás también de 2023 con 31.89%).

Ahora analicemos, si los tres hubieran dado el pico exacto en 2022, sería sospechoso como si estuviéramos viendo lo que queríamos ver. Que maíz se adelante a 2021 da una pregunta real para investigar (2021 fue el año de plena recuperación pospandemia, con cuellos de botella logísticos globales y una demanda de maíz de China que venía disparada) (China había arrancado a comprar maíz de EE.UU. en volúmenes históricos en 2020-2021, tras un brote de peste porcina que la obligó a reconstruir su stock de alimento balanceado). Esto es un dato verificable y le da al proyecto un hallazgo propio, no calcado del "todo fue por la sequía de 2022".

Trigo llama la atención por otro motivo su salto en 2022 es el más extremo de los tres cultivos (52% vs un rango normal de 25-30% en los demás años) coincide con que trigo es, además de la sequía argentina, el cultivo más golpeado directamente por la guerra Rusia-Ucrania, porque ambos países son exportadores gigantes de trigo (mucho más relevantes en trigo que en soja o maíz). Ahí sí el shock geopolítico pega más fuerte y de forma más específica que en los otros dos.
La conclusión de negocio, en las 4 partes

Concluyendo esta etapa del analisis, el trigo tuvo la volatilidad más extrema del período (52% en 2022, casi el doble de su nivel normal); soja tocó su pico también en 2022 (29%); maíz, en cambio, fue más volátil en 2021 (36%) que en 2022. Por qué el trigo se ve más golpeado por su exposición directa a la guerra Rusia-Ucrania (grandes exportadores de trigo); maíz muestra su pico un año antes, coincidiendo con la demanda excepcional de China en 2021 tras la crisis de peste porcina.

La necesidad de cobertura no es igual todos los años ni igual entre cultivos un área de riesgo debería monitorear trigo con más atención en contextos de tensión geopolítica en el Mar Negro, y maíz ante señales de demanda extraordinaria de China, más que asumir que "siempre es más riesgoso en años de sequía local".

La volatilidad anualizada asume que el comportamiento diario se mantiene constante todo el año, lo cual es una simplificación en la práctica, la volatilidad no es pareja dentro del año (suele concentrarse en meses puntuales, como se puido notar en la Pregunta 1).

# Análisis (PREGUNTA N°5)

## ¿La cobertura con futuros hubiera protegido al productor en la campaña 2022?

Si un productor hubiera tomado un futuro de soja antes de la cosecha, en la campaña de la sequía (2022), ¿se hubiera protegido o hubiera perdido plata comparado con vender sin cobertura?





Un productor que todavía no cosechó, pero ya sabe más o menos cuánto va a producir, puede "vender hoy" ese grano a un precio fijado, aunque la entrega sea recién en unos meses. Si el precio de mercado después baja, el productor ganó vendió más caro de lo que hubiera conseguido esperando. Si el precio sube, el productor "perdió" esa suba porque ya se comprometió a un precio menor. La cobertura no es apostar a ganar más, es sacarse la incertidumbre de encima, a cambio de resignar la posibilidad de un precio mejor.

Ahora vamos a  simular con tus datos, se elije una fecha de "toma de cobertura" por ejemplo, unos meses antes de la cosecha de soja (en Argentina la cosecha gruesa es abril-mayo, así que se podria simular que el productor toma el futuro en, digamos, enero de esa campaña).
Ese precio de esa fecha es el "precio cubierto" el que el productor se aseguró. Se Compara ese precio contra el precio que efectivamente hubo en la fecha de cosecha/venta real (mayo, por ejemplo). Si el precio de cosecha fue menor al precio cubierto → la cobertura protegió (ganó el que se cubrió). Si el precio de cosecha fue mayor → la cobertura "costó" esa diferencia (perdió el que se cubrió, comparado con no haberse cubierto).

Esta campaña la elijo, la campaña 2022 puntualmente porque es la que se ha venido identificando que fue el año de mayor volatilidad y la sequía más severa es el caso de uso más ilustrativo para mostrar si la cobertura realmente cumple su función en un año de estrés, que es cuando más importa.


### Definir la campaña y las dos fechas clave (cobertura vs cosecha)

In [71]:
soja_2022 = precios_diarios_todos[
    (precios_diarios_todos["cultivo"] == "soja") &
    (precios_diarios_todos["anio"] == 2022)
].sort_values("fecha")

precio_cobertura = soja_2022[soja_2022["mes"] == 1]["precio_cierre"].iloc[0]
fecha_cobertura = soja_2022[soja_2022["mes"] == 1]["fecha"].iloc[0]

precio_cosecha = soja_2022[soja_2022["mes"] == 5]["precio_cierre"].iloc[-1]
fecha_cosecha = soja_2022[soja_2022["mes"] == 5]["fecha"].iloc[-1]

print(f"Precio cobertura ({fecha_cobertura.date()}): {precio_cobertura:.2f}")
print(f"Precio cosecha ({fecha_cosecha.date()}): {precio_cosecha:.2f}")

Precio cobertura (2022-01-03): 1344.00
Precio cosecha (2022-05-31): 1683.25


### Elegir el precio de cobertura (enero) y el precio de cosecha (mayo)

In [72]:
precio_cobertura = soja_2022[soja_2022["fecha"].dt.month == 1]["precio_cierre"].iloc[0]
fecha_cobertura = soja_2022[soja_2022["fecha"].dt.month == 1]["fecha"].iloc[0]

precio_cosecha = soja_2022[soja_2022["fecha"].dt.month == 5]["precio_cierre"].iloc[-1]
fecha_cosecha = soja_2022[soja_2022["fecha"].dt.month == 5]["fecha"].iloc[-1]

print(f"Precio cobertura ({fecha_cobertura.date()}): {precio_cobertura:.2f}")
print(f"Precio cosecha ({fecha_cosecha.date()}): {precio_cosecha:.2f}")

Precio cobertura (2022-01-03): 1344.00
Precio cosecha (2022-05-31): 1683.25


In [73]:
diferencia_absoluta = precio_cosecha - precio_cobertura
diferencia_pct = (diferencia_absoluta / precio_cobertura) * 100

print(f"Diferencia: {diferencia_absoluta:.2f} centavos/bushel ({diferencia_pct:.1f}%)")

if diferencia_absoluta > 0:
    print("La cobertura resignó una suba de precio: hubiera sido mejor NO cubrirse.")
else:
    print("La cobertura protegió de una baja de precio: hubiera sido mejor cubrirse.")

Diferencia: 339.25 centavos/bushel (25.2%)
La cobertura resignó una suba de precio: hubiera sido mejor NO cubrirse.


## Conclusión: ¿la cobertura hubiera protegido al productor?

Punto clave, la cobertura no se evalúa mirando hacia atrás con el resultado ya conocido se toma en enero, sin saber qué va a pasar en mayo. En enero de 2022, con la sequía ya empezando a manifestarse y la guerra Rusia-Ucrania recién estallando, un productor no tenía forma de saber que el precio iba a subir 25% más. Lo que la cobertura le daba en enero era certeza sabía exactamente cuánto iba a cobrar, podía planificar sus costos, pagar arrendamientos, tomar créditos con ese número fijo. Eso tiene un valor en sí mismo, incluso si en este caso puntual "no convino" en términos de precio final. Es la misma lógica que un seguro de auto si no chocaste en todo el año, "perdiste" la plata de la prima pero eso no significa que fue una mala decisión asegurarte, porque cuando lo contrataste no sabías si ibas a chocar o no. La cobertura de precio deberia funcionar igual es un seguro contra la incertidumbre, no una apuesta a acertar la dirección del mercado.

Finalmente en la campaña de soja 2022, cubrirse en enero resultó en resignar un 25.2% de suba respecto al precio de cosecha en mayo la cobertura, en este caso puntual, "costó" en términos de resultado final.

Por qué la sequía y la guerra Rusia-Ucrania dispararon el precio después de enero, un escenario que no era predecible al momento de tomar la cobertura.

Esto implica que la cobertura no debe evaluarse por si "acertó" el precio final, sino por el valor de la certeza que le dio al productor en el momento de decidir permite planificar costos y financiamiento sin depender de hacia dónde se mueva el mercado. Una gestión de riesgo madura no mide el éxito de la cobertura por si ganó o perdió contra el precio final, sino por si redujo la exposición a un escenario adverso que, al momento de decidir, era un riesgo real.

Esta es una simulación simplificada con precio de cierre diario un futuro real de BCR/MATBA tiene vencimientos específicos, requiere márgenes de garantía, y el productor podría haber tomado cobertura parcial (no del 100% de su producción) para balancear protección con la posibilidad de capturar parte de la suba. Además, esta es una sola campaña para una conclusión más sólida habría que repetir este ejercicio en varios años, incluyendo campañas donde el precio bajó, para ver el patrón completo del "seguro" funcionando en ambos sentidos.

# Cierre 1 HALLAZGOS PRINCIPALES CONSOLIDADOS

In [74]:
from IPython.display import display, Markdown

hallazgos = """
## Hallazgos principales

**1. Estacionalidad:** los tres cultivos muestran caída sistemática en julio-agosto
(hasta -7.6% en maíz) y rebote en septiembre (+5.8% a +6.2%), coincidiendo con la
definición de rendimiento y el inicio de cosecha en el hemisferio norte.

**2. Correlación entre cultivos:** correlación moderada en los tres pares (0.34 a 0.54).
Maíz-soja y maíz-trigo comparten más movimiento (0.54) que soja-trigo (0.34),
consistente con que maíz y soja comparten campaña agrícola en Argentina.

**3. Producción argentina vs precio internacional:** el rendimiento argentino
correlaciona fuerte con el precio de Chicago (-0.65 a -0.83), pero la producción
total mucho menos (-0.08 a -0.64) — evidencia de que el rendimiento captura un
factor climático regional/global compartido, no un efecto exclusivo de Argentina
sobre el mercado mundial.

**4. Volatilidad:** trigo alcanzó su pico histórico en 2022 (52% anualizado, casi
el doble de su nivel normal), coincidiendo con la guerra Rusia-Ucrania. Maíz, en
cambio, fue más volátil en 2021 (36%), coincidiendo con la demanda excepcional
de China tras la crisis de peste porcina.

**5. Cobertura con futuros:** en la campaña de soja 2022, cubrirse en enero resignó
un 25.2% de suba respecto al precio de cosecha en mayo. La cobertura debe evaluarse
por la certeza que aporta al momento de decidir, no por si acertó la dirección del
precio final.
"""

display(Markdown(hallazgos))


## Hallazgos principales

**1. Estacionalidad:** los tres cultivos muestran caída sistemática en julio-agosto
(hasta -7.6% en maíz) y rebote en septiembre (+5.8% a +6.2%), coincidiendo con la
definición de rendimiento y el inicio de cosecha en el hemisferio norte.

**2. Correlación entre cultivos:** correlación moderada en los tres pares (0.34 a 0.54).
Maíz-soja y maíz-trigo comparten más movimiento (0.54) que soja-trigo (0.34),
consistente con que maíz y soja comparten campaña agrícola en Argentina.

**3. Producción argentina vs precio internacional:** el rendimiento argentino
correlaciona fuerte con el precio de Chicago (-0.65 a -0.83), pero la producción
total mucho menos (-0.08 a -0.64) — evidencia de que el rendimiento captura un
factor climático regional/global compartido, no un efecto exclusivo de Argentina
sobre el mercado mundial.

**4. Volatilidad:** trigo alcanzó su pico histórico en 2022 (52% anualizado, casi
el doble de su nivel normal), coincidiendo con la guerra Rusia-Ucrania. Maíz, en
cambio, fue más volátil en 2021 (36%), coincidiendo con la demanda excepcional
de China tras la crisis de peste porcina.

**5. Cobertura con futuros:** en la campaña de soja 2022, cubrirse en enero resignó
un 25.2% de suba respecto al precio de cosecha en mayo. La cobertura debe evaluarse
por la certeza que aporta al momento de decidir, no por si acertó la dirección del
precio final.


# Cierre 2 ALCANCE Y LIMITACIONES

In [75]:
limitaciones = """
## Alcance y limitaciones

- Los precios corresponden al mercado de futuros de Chicago (CME), referencia
  internacional. No se incluyó precio de mercado local (MATBA/Rosario) por no
  contar con una fuente pública abierta equivalente para el período analizado.

- Las correlaciones de las Preguntas 2 y 3 miden relación lineal (Pearson) y
  no implican causalidad. En 2022 coinciden sequía regional y guerra Rusia-Ucrania,
  dos causas simultáneas que este análisis no puede separar entre sí.

- La Pregunta 3 usa 10 observaciones anuales (2015-2024) — una muestra chica
  para conclusiones estadísticamente robustas. El hallazgo es una asociación
  observada en el período, no una ley general.

- La simulación de cobertura (Pregunta 5) usa precio de cierre diario sobre una
  sola campaña. No modela vencimientos de contrato, márgenes de garantía, ni
  cobertura parcial — mecánicas reales del mercado de futuros que un caso de
  uso operativo sí debería incorporar.

- El análisis cubre soja, maíz y trigo a nivel nacional. No incluye apertura
  por provincia/departamento (datos disponibles pero fuera del alcance de
  este proyecto) ni el mercado ganadero.
"""

display(Markdown(limitaciones))


## Alcance y limitaciones

- Los precios corresponden al mercado de futuros de Chicago (CME), referencia
  internacional. No se incluyó precio de mercado local (MATBA/Rosario) por no
  contar con una fuente pública abierta equivalente para el período analizado.

- Las correlaciones de las Preguntas 2 y 3 miden relación lineal (Pearson) y
  no implican causalidad. En 2022 coinciden sequía regional y guerra Rusia-Ucrania,
  dos causas simultáneas que este análisis no puede separar entre sí.

- La Pregunta 3 usa 10 observaciones anuales (2015-2024) — una muestra chica
  para conclusiones estadísticamente robustas. El hallazgo es una asociación
  observada en el período, no una ley general.

- La simulación de cobertura (Pregunta 5) usa precio de cierre diario sobre una
  sola campaña. No modela vencimientos de contrato, márgenes de garantía, ni
  cobertura parcial — mecánicas reales del mercado de futuros que un caso de
  uso operativo sí debería incorporar.

- El análisis cubre soja, maíz y trigo a nivel nacional. No incluye apertura
  por provincia/departamento (datos disponibles pero fuera del alcance de
  este proyecto) ni el mercado ganadero.


# Resumen KPIs ejecutivos

In [76]:
from IPython.display import display, HTML

def tarjeta_kpi(titulo, valor, contexto, color):
    return f"""
    <div style="
        flex: 1;
        min-width: 180px;
        background: white;
        border: 2px solid {color};
        border-radius: 14px;
        padding: 18px;
        text-align: center;
        box-shadow: 0 4px 10px rgba(0,0,0,0.06);
    ">
        <div style="font-size: 12px; color: #64748B; text-transform: uppercase; letter-spacing: 1px;">
            {titulo}
        </div>
        <div style="font-size: 26px; font-weight: 700; color: {color}; margin: 8px 0;">
            {valor}
        </div>
        <div style="font-size: 12px; color: #475569;">
            {contexto}
        </div>
    </div>
    """

kpis_html = f"""
<div style="display: flex; gap: 14px; flex-wrap: wrap; margin: 20px 0;">
    {tarjeta_kpi("Mejor mes de venta", "Septiembre", "+5.8% a +6.2% promedio histórico", "#0F766E")}
    {tarjeta_kpi("Diversificación", "Soja-Trigo 0.34", "correlación más baja entre pares", "#1E3A5F")}
    {tarjeta_kpi("Argentina, tomador de precio", "-0.83", "correlación rendimiento-precio (maíz)", "#D4A72C")}
    {tarjeta_kpi("Año más volátil", "Trigo 2022", "52% anualizado", "#B91C1C")}
    {tarjeta_kpi("Cobertura soja 2022", "-25.2%", "vs. no cubrirse, campaña puntual", "#E07A5F")}
</div>
"""

display(HTML(kpis_html))

# Pipeline unificado

In [78]:
from IPython.display import display, HTML

def tarjeta_flujo(numero, titulo, descripcion, color):
    return f"""
    <div style="
        flex: 1;
        min-width: 170px;
        min-height: 140px;
        background: white;
        border: 2px solid {color};
        border-radius: 16px;
        padding: 28px 14px 16px 14px;
        text-align: center;
        box-shadow: 0 4px 10px rgba(0,0,0,0.06);
        position: relative;
    ">
        <div style="
            position: absolute; top: -14px; left: 50%; transform: translateX(-50%);
            background: {color}; color: white; width: 28px; height: 28px;
            border-radius: 50%; display: flex; align-items: center; justify-content: center;
            font-size: 13px; font-weight: 700;
        ">{numero}</div>
        <div style="font-size: 13px; font-weight: 700; color: {color}; margin-bottom: 8px;">
            {titulo}
        </div>
        <div style="font-size: 11.5px; color: #475569; line-height: 1.4;">
            {descripcion}
        </div>
    </div>
    """

pipeline_html = f"""
<div style="
    background: #F8FAFC;
    border-radius: 20px;
    padding: 28px 24px;
    margin: 20px 0;
">
    <div style="text-align:center; margin-bottom: 22px;">
        <div style="font-size: 20px; font-weight: 700; color: #1E3A5F;">
            Pipeline del proyecto
        </div>
        <div style="font-size: 12.5px; color: #64748B; margin-top: 4px;">
            ETL lineal hasta el dato limpio, luego ramificado según cada pregunta de negocio
        </div>
    </div>

    <div style="font-size: 11px; font-weight: 700; color: #1E3A5F; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 10px;">
        Tronco común — ETL
    </div>
    <div style="display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 26px;">
        {tarjeta_flujo("1", "EXTRACT", "yfinance (API, precios diarios Chicago) + CSV públicos MAGyP (producción nacional)", "#1E3A5F")}
        {tarjeta_flujo("2", "EDA", "Rango de fechas, nulos, duplicados, valores imposibles — validación de calidad antes de transformar", "#0F766E")}
        {tarjeta_flujo("3", "TRANSFORM base", "Aplanar MultiIndex, unificar nombres de columna, filtrar 2015-2024 comparable", "#D4A72C")}
        {tarjeta_flujo("4", "LOAD", "Datasets limpios guardados en data/processed/ — reutilizables sin rehacer el pipeline", "#475569")}
    </div>

    <div style="text-align:center; font-size:20px; color:#94A3B8; margin-bottom: 10px;">↓</div>

    <div style="font-size: 11px; font-weight: 700; color: #0F766E; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 10px;">
        Ramas — transformación específica por pregunta de negocio
    </div>
    <div style="display: flex; gap: 10px; flex-wrap: wrap;">
        {tarjeta_flujo("5a", "Estacionalidad", "Agregación diario→mensual + pct_change() agrupado por cultivo", "#0F766E")}
        {tarjeta_flujo("5b", "Correlación", "pivot() a formato ancho (una columna por cultivo) + .corr()", "#0F766E")}
        {tarjeta_flujo("5c", "Producción vs precio", "Agregación diario→anual + merge() con MAGyP por cultivo y año", "#0F766E")}
        {tarjeta_flujo("5d", "Volatilidad", "groupby cultivo-año + std() de retornos, anualizada con √252", "#0F766E")}
        {tarjeta_flujo("5e", "Cobertura", "Recorte de campaña puntual + comparación de dos fechas de precio", "#0F766E")}
    </div>
</div>
"""

display(HTML(pipeline_html))

# Para mi colega Técnico
## Nota metodológica pipeline ramificado

El proyecto sigue un ETL clásico (Extract → Transform → Load) hasta el dataset limpio, pero a partir de ahí el pipeline se ramifica, cada pregunta de negocio requiere una transformación distinta sobre el mismo dato base (agregación temporal, reshape a formato ancho, merge entre fuentes, o recorte de ventana temporal), en vez de forzar una única tabla maestra que sirva para las cinco preguntas a la vez.

Esta decisión de diseño evita una tabla intermedia sobrecargada de columnas que solo aplican a una pregunta puntual, y mantiene cada transformación trazable hasta la pregunta que la originó.

display(Markdown(texto_metodologico))